# RedPill Verification Process

This notebook walks through the non-streaming example from `signature_verifier.py` one step at a time.

In [ ]:
import json
import os
import secrets
from hashlib import sha256

import requests
from dotenv import load_dotenv
from eth_account import Account
from eth_account.messages import encode_defunct

from attestation_verifier import (
    check_gpu,
    check_report_data,
    check_tdx_quote,
    show_sigstore_provenance,
)

load_dotenv()

API_KEY = os.environ.get("API_KEY", "")
BASE_URL = "https://api.redpill.ai"
MODEL = "phala/qwen3-vl-30b-a3b-instruct"

if not API_KEY:
    raise ValueError("Set API_KEY in your environment or .env before running the notebook.")


def sha256_text(text):
    return sha256(text.encode()).hexdigest()


def recover_signer(text, signature):
    message = encode_defunct(text=text)
    return Account.recover_message(message, signature=signature)


## Step 1: Chat via API

In [ ]:
body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Hello, what is the capital of France?"}],
    "stream": False,
    "max_tokens": 1000,
}
body_json = json.dumps(body)

print("Sending request:")
print(json.dumps(body, indent=2))

response = requests.post(
    f"{BASE_URL}/v1/chat/completions",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    },
    data=body_json,
    timeout=30,
)
response.raise_for_status()

payload = response.json()
chat_id = payload["id"]
response_text = response.text

print("Chat ID:", chat_id)
print("API response:")
print(json.dumps(payload, indent=2))


## Step 2.1: Get TEE's signature on (request, response) pair

In [ ]:
signature_response = requests.get(
    f"{BASE_URL}/v1/signature/{chat_id}?model={MODEL}",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
signature_response.raise_for_status()
signature_payload = signature_response.json()

print(json.dumps(signature_payload, indent=2))


## Step 2.2: Verify the sig

In [ ]:
request_hash = sha256_text(body_json)
response_hash = sha256_text(response_text)

hashed_text = signature_payload["text"]
request_hash_server, response_hash_server = hashed_text.split(":")

print("Request hash matches:", request_hash == request_hash_server)
print("Response hash matches:", response_hash == response_hash_server)

signature = signature_payload["signature"]
signing_address = signature_payload["signing_address"]
recovered = recover_signer(hashed_text, signature)

# print("Claimed signing address:", signing_address)
print("Recovered signing address:", recovered)
print("Signature valid:", recovered.lower() == signing_address.lower())


## Step 3: Verify the signer runs in TDX TEE

### Step 3.1: Fetch a fresh attestation for the signer

In [ ]:
nonce = secrets.token_hex(32)
attestation_response = requests.get(
    f"{BASE_URL}/v1/attestation/report?model={MODEL}&nonce={nonce}&signing_address={signing_address}",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
attestation_response.raise_for_status()
report = attestation_response.json()

if "all_attestations" in report:
    attestation = next(
        item for item in report["all_attestations"]
        if item["signing_address"].lower() == signing_address.lower()
    )
else:
    attestation = report

with open("attestation-report.json", "w", encoding="utf-8") as file:
    json.dump(attestation, file, indent=2)


print("Has Intel quote:", "intel_quote" in attestation)
print("Has NVIDIA payload:", "nvidia_payload" in attestation)

### Step 3.2: Verify Intel TDX quote & Check that report data binds the signer and nonce

`check_tdx_quote(attestation)` does three things:

1. It takes the raw `intel_quote` field from the attestation we fetched from Redpill.
2. It sends that quote to Phala's TDX verification endpoint at `https://cloud-api.phala.network/api/v1/attestations/verify`.
3. It reads the verifier response and prints whether the quote was marked as verified, along with any message returned by the verifier.

The returned `intel_result` object also contains the decoded quote body, which is useful for teaching because it exposes measurements like `mrtd`, `mrconfig`, and `reportdata` that we inspect in the next steps.

In [ ]:
intel_result = check_tdx_quote(attestation)

quote_body = intel_result["quote"]["body"]
print("mr_td:", quote_body.get("mrtd"))
print("mr_config:", quote_body.get("mrconfig"))
print("report_data:", quote_body.get("reportdata"))

report_data_check = check_report_data(attestation, nonce, intel_result)

## Step 6: Verify the GPU attestation

`check_gpu(attestation, nonce)` follows the GPU evidence path in `attestation_verifier.py`:

1. It reads the `nvidia_payload` field from the attestation and parses it as JSON.
2. It compares the GPU payload's `nonce` against the nonce we generated when we requested the attestation. This checks freshness and helps prevent replay.
3. It sends the whole GPU evidence payload to NVIDIA's NRAS verifier at `https://nras.attestation.nvidia.com/v3/attest/gpu`.
4. It extracts the JWT token from NVIDIA's response.
5. It decodes the JWT payload and reads the `x-nvidia-overall-att-result` field, which is the overall verdict for the GPU attestation.

So this step teaches two different ideas at once: first, the attestation bundle is tied to our nonce, and second, NVIDIA's verifier independently confirms whether the GPU evidence is valid.

In [ ]:
gpu_check = check_gpu(attestation, nonce)
gpu_check


## Step 7: Check Sigstore provenance for the attested containers

`show_sigstore_provenance(attestation)` explains the software supply-chain side of the attestation:

1. It looks inside `attestation["info"]["tcb_info"]` to find the `app_compose` data.
2. It scans that compose content for container image digests that match the `@sha256:...` pattern.
3. It deduplicates those digests so the same image is only checked once.
4. For each digest, it builds a Sigstore search URL like `https://search.sigstore.dev/?hash=sha256:...`.
5. It sends an HTTP `HEAD` request to each URL and reports whether the provenance entry is accessible.

This step does not prove the enclave hardware is genuine by itself. Instead, it helps teach how the attestation points to the exact container images used by the workload, and how those images can be traced back through Sigstore provenance records.

In [ ]:
print(json.dumps(attestation["info"]["tcb_info"], indent=2))